In [ ]:
# import shutil
# import os

# def delete_directory(path):
#     if not os.path.exists(path):
#         print(f"Thư mục '{path}' không tồn tại.")
#         return

#     print(f"Bạn có chắc chắn muốn xoá toàn bộ thư mục và tất cả nội dung của nó tại: {path}?")
#     confirmation = input("Nhập 'yes' để xác nhận: ").strip().lower()

#     if confirmation == 'yes':
#         try:
#             shutil.rmtree(path)
#             print(f"Thư mục '{path}' đã được xoá thành công.")
#         except OSError as e:
#             print(f"Lỗi: Không thể xoá thư mục {path}. {e}")
#     else:
#         print("Hành động xoá đã bị hủy.")

# # Nhập đường dẫn thư mục bạn muốn xoá
# directory_to_delete = input("Vui lòng nhập đường dẫn thư mục cần xoá: ")

# delete_directory(directory_to_delete)

# BGE-M3 Vietnamese Medical Dense Embedding — Full Merged LoRA Pipeline

Notebook này là bản sửa từ notebook đã train thành công nhưng `load_best_model_at_end=True` fail khi load best checkpoint.

## Sửa chính trong bản này

Không dùng `Trainer.load_best_model_at_end` nữa vì PEFT-wrapped `SentenceTransformer` checkpoint bị lệch key prefix:

- checkpoint có key kiểu `0.auto_model...`
- PEFT-wrapped model hiện tại cần key kiểu `0.auto_model.base_model.model...`

Thay vào đó notebook tự lưu **LoRA adapter state tốt nhất** theo dev `cosine_ndcg@10`:

1. Sau mỗi eval step, nếu dev metric tốt hơn best cũ thì lưu:
   - `best_adapter_state.pt`
   - `best_adapter_info.json`
2. Sau training, evaluate final model thêm một lần.
3. Nếu final model tốt hơn best đã lưu thì ghi đè best.
4. Load lại `best_adapter_state.pt` vào model.
5. Merge LoRA best adapter vào base model.
6. Save full merged SentenceTransformers artifact.
7. Reload local, push Hub, reload Hub.

## Kết quả mong muốn

Model cuối cùng phải là:

```text
best LoRA adapter by dev metric
→ merged into BAAI/bge-m3 base
→ full SentenceTransformers artifact
→ not adapter-only
```

Chạy trên fresh runtime có GPU.


In [ ]:
# 01_install_clean_environment
# Chạy trên fresh runtime. Không import torch/transformers/sentence_transformers trước cell này.

import sys
import subprocess

def run_cmd(cmd, check=True, capture=False):
    print("$", " ".join(cmd))
    if capture:
        result = subprocess.run(
            cmd,
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        print(result.stdout)
        if check and result.returncode != 0:
            raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
        return result
    return subprocess.run(cmd, check=check)

# Gỡ các package có thể gây lỗi pipeline.
run_cmd(
    [
        sys.executable, "-m", "pip", "uninstall", "-y",
        "unsloth", "unsloth_zoo", "trl", "torchao", "torchao",
        "sentence-transformers", "transformers", "peft",
        "datasets", "accelerate", "huggingface_hub",
    ],
    check=False,
)

# Cài stack cố định, không để pip tự chọn PEFT/Transformers mới nhất.
run_cmd([
    sys.executable, "-m", "pip", "install", "-U", "--quiet",
    "sentence-transformers[train]==5.1.2",
    "transformers==4.57.1",
    "peft==0.17.1",
    "datasets==4.4.1",
    "accelerate==1.11.0",
    "huggingface-hub==0.36.0",
    "safetensors",
    "python-dotenv",
    "hf_xet",
    "jedi",
])

# pip check chỉ cảnh báo, không chặn notebook.
pip_check = run_cmd([sys.executable, "-m", "pip", "check"], check=False, capture=True)

if pip_check.returncode != 0:
    print("=" * 80)
    print("WARNING: pip check phát hiện dependency conflict.")
    print("Nếu conflict không liên quan đến torch / transformers / sentence-transformers / peft / datasets / accelerate / huggingface_hub thì có thể chạy tiếp.")
    print("=" * 80)
else:
    print("pip check passed.")

print("Install completed.")

In [ ]:

# 02_imports_versions_and_global_config

import os
import re
import gc
import json
import math
import shutil
import random
import inspect
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

import transformers
import datasets
import peft
import sentence_transformers
import huggingface_hub

from datasets import load_dataset, Dataset, concatenate_datasets
from peft import LoraConfig, TaskType, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import cos_sim
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from huggingface_hub import HfApi, login, whoami
from dotenv import load_dotenv
from transformers import TrainerCallback

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

assert "unsloth" not in globals(), "Do not use Unsloth in this notebook."

# ----------------------------
# Core config
# ----------------------------
BASE_MODEL = "BAAI/bge-m3"
DATASET_ID = "nqd-301125/medical_data"
REPO_ID = "nqd-301125/bge-m3-medical-vi-dense"

SEED = 3407

MAX_SEQ_LENGTH = 6000

# Colab A100/L4-large target. Nếu OOM, giảm TRAIN_BATCH_SIZE xuống 128 hoặc 64.
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 256
GRADIENT_ACCUMULATION_STEPS = 1

NUM_EPOCHS = 3

# Cấu hình an toàn hơn so với LR 1e-4 trong notebook reviewed.
LEARNING_RATE = 3e-5
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ["query", "key", "value", "dense"]

# Dataset cleanup/split
DROP_DUPLICATE_ANCHORS = True
DROP_DUPLICATE_POSITIVES = True
TRAIN_RATIO = 0.92
DEV_RATIO = 0.04
TEST_RATIO = 0.04

# Optional baseline eval. Giữ True để có mốc so sánh trước train.
RUN_BASELINE_EVAL = True

# Push config
PUSH_TO_HUB = True
DELETE_REMOTE_FILES_BEFORE_UPLOAD = True

OUTPUT_ROOT = Path("/content/bge_m3_medical_vi_dense_run")
TRAIN_OUTPUT_DIR = OUTPUT_ROOT / "checkpoints"
LOCAL_MERGED_DIR = OUTPUT_ROOT / "bge-m3-medical-vi-dense-full-merged"
METRICS_DIR = OUTPUT_ROOT / "metrics"
HUB_RELOAD_CACHE = OUTPUT_ROOT / "hub_reload_cache"
EXPORT_ZIP_PATH = OUTPUT_ROOT / "bge-m3-medical-vi-dense-full-merged.zip"
BEST_ADAPTER_DIR = OUTPUT_ROOT / "best_lora_adapter_state"
BEST_ADAPTER_STATE_PATH = BEST_ADAPTER_DIR / "best_adapter_state.pt"
BEST_ADAPTER_INFO_PATH = BEST_ADAPTER_DIR / "best_adapter_info.json"

os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    TORCH_DTYPE = torch.bfloat16
    USE_BF16 = True
    USE_FP16 = False
elif torch.cuda.is_available():
    TORCH_DTYPE = torch.float16
    USE_BF16 = False
    USE_FP16 = True
else:
    TORCH_DTYPE = torch.float32
    USE_BF16 = False
    USE_FP16 = False

print("TORCH_DTYPE:", TORCH_DTYPE)
print("USE_BF16:", USE_BF16, "USE_FP16:", USE_FP16)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 03_auth_huggingface
# Bắt buộc nhập HF token thủ công. Không tự load từ env, .env, Colab Secret hay cache.

import os
from getpass import getpass
from huggingface_hub import login, whoami, logout

# Xóa các biến môi trường token nếu có, để tránh tự dùng token cũ.
for key in [
    "HF_TOKEN",
    "HUGGINGFACE_HUB_TOKEN",
    "HUGGING_FACE_HUB_TOKEN",
    "HUGGINGFACEHUB_API_TOKEN",
]:
    if key in os.environ:
        del os.environ[key]

# Logout token cache cũ nếu có.
try:
    logout()
    print("Logged out cached Hugging Face token.")
except Exception as e:
    print("No cached Hugging Face login to logout, or logout skipped:", repr(e))

# Bắt buộc nhập thủ công.
HF_TOKEN = getpass("Nhập Hugging Face token thủ công: ").strip()

if PUSH_TO_HUB:
    if not HF_TOKEN:
        raise RuntimeError("Missing HF_TOKEN.")

    # Login bằng đúng token vừa nhập.
    login(token=HF_TOKEN, add_to_git_credential=False)

    # whoami cũng dùng đúng token vừa nhập, không dùng cache.
    user_info = whoami(token=HF_TOKEN)
    print("Logged in as:", user_info.get("name") or user_info)
else:
    print("PUSH_TO_HUB=False, skip HF login.")

In [ ]:

# 04_utilities

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def normalize_text(s):
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def stable_hash(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

def load_sentence_model(model_name_or_path, max_seq_length=MAX_SEQ_LENGTH, trust_remote_code=True):
    kwargs = dict(trust_remote_code=trust_remote_code)

    # ST 5.x supports model_kwargs. Older versions may not.
    try:
        model = SentenceTransformer(
            model_name_or_path,
            model_kwargs={"dtype": TORCH_DTYPE},
            **kwargs,
        )
    except TypeError:
        model = SentenceTransformer(model_name_or_path, **kwargs)
        if TORCH_DTYPE != torch.float32:
            model = model.to(TORCH_DTYPE)

    model.max_seq_length = max_seq_length

    if torch.cuda.is_available():
        model = model.cuda()

    return model

def smoke_test_model(model, expected_dim=1024, name="model"):
    texts = [
        "Đau đầu kéo dài có nguy hiểm không?",
        "Trẻ bị sốt cao nên xử lý như thế nào?",
    ]
    emb = model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
        batch_size=2,
    )
    print(name, "embedding shape:", emb.shape)
    print(name, "finite:", np.isfinite(emb).all())
    print(name, "norm:", np.linalg.norm(emb, axis=1))

    assert emb.shape == (2, expected_dim), f"{name}: expected shape (2,{expected_dim}), got {emb.shape}"
    assert np.isfinite(emb).all(), f"{name}: embedding contains NaN/Inf"
    return emb

def find_ndcg10_key(metrics):
    keys = list(metrics.keys())
    candidates = [k for k in keys if k.endswith("cosine_ndcg@10")]
    if not candidates:
        candidates = [k for k in keys if "ndcg@10" in k and "cosine" in k]
    if not candidates:
        raise KeyError(f"Cannot find cosine_ndcg@10 in metric keys: {keys}")
    return candidates[0]

def summarize_ir_metrics(metrics):
    out = {}
    for k, v in metrics.items():
        if any(x in k for x in [
            "accuracy@1", "accuracy@3", "accuracy@5", "accuracy@10",
            "precision@1", "precision@3", "precision@5", "precision@10",
            "recall@1", "recall@3", "recall@5", "recall@10",
            "ndcg@10", "mrr@10", "map@100"
        ]):
            if isinstance(v, (int, float, np.floating)):
                out[k] = float(v)
    return out


def extract_metric_value(metrics, metric_name):
    """Return metric value from Trainer/evaluator metrics with robust key matching."""
    if metrics is None:
        return None

    candidate_keys = []
    if metric_name:
        candidate_keys.append(metric_name)
        if metric_name.startswith("eval_"):
            candidate_keys.append(metric_name[len("eval_"):])
        else:
            candidate_keys.append("eval_" + metric_name)

    # Prefer exact keys first.
    for key in candidate_keys:
        if key in metrics:
            return float(metrics[key])

    # Fallback: find cosine_ndcg@10.
    for key, value in metrics.items():
        if key.endswith("cosine_ndcg@10"):
            return float(value)

    # Broader fallback.
    for key, value in metrics.items():
        if "ndcg@10" in key and "cosine" in key:
            return float(value)

    return None


def save_current_peft_adapter_state(st_model, output_dir, metric_name, score, step=None, epoch=None, source="eval"):
    """Save only the LoRA adapter state. Base model is frozen, so adapter state is enough."""
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    auto_model = st_model[0].auto_model
    assert hasattr(auto_model, "peft_config"), "Current auto_model is not a PEFT model."

    adapter_state = get_peft_model_state_dict(auto_model)
    adapter_state_cpu = {
        k: v.detach().cpu()
        for k, v in adapter_state.items()
    }

    state_path = output_dir / "best_adapter_state.pt"
    torch.save(adapter_state_cpu, state_path)

    # Save PEFT config for audit/recovery. Loading in this notebook uses the existing same config.
    try:
        auto_model.peft_config["default"].save_pretrained(str(output_dir))
    except Exception as e:
        print("Could not save PEFT config via save_pretrained:", repr(e))

    info = {
        "metric_name": metric_name,
        "score": float(score) if score is not None else None,
        "step": int(step) if step is not None else None,
        "epoch": float(epoch) if epoch is not None else None,
        "source": source,
        "state_path": str(state_path),
    }
    save_json(info, output_dir / "best_adapter_info.json")

    print(f"Saved best adapter state: score={info['score']} step={info['step']} epoch={info['epoch']} source={source}")
    print("Best adapter state path:", state_path)

    return info


def load_peft_adapter_state_into_model(st_model, state_path, adapter_name="default"):
    """Load saved LoRA adapter state back into the current PEFT-wrapped SentenceTransformer."""
    state_path = Path(state_path)
    assert state_path.exists(), f"Missing best adapter state file: {state_path}"

    auto_model = st_model[0].auto_model
    assert hasattr(auto_model, "peft_config"), "Current auto_model is not a PEFT model."

    adapter_state = torch.load(state_path, map_location="cpu")

    try:
        load_result = set_peft_model_state_dict(
            auto_model,
            adapter_state,
            adapter_name=adapter_name,
        )
    except TypeError:
        load_result = set_peft_model_state_dict(
            auto_model,
            adapter_state,
        )

    print("Loaded PEFT adapter state from:", state_path)
    print("PEFT load result:", load_result)

    if torch.cuda.is_available():
        st_model = st_model.cuda()

    return st_model


In [ ]:

# 05_load_clean_split_dataset

raw = load_dataset(DATASET_ID, split="train")
print(raw)
print("columns:", raw.column_names)

required_cols = {"anchor", "positive"}
missing = required_cols - set(raw.column_names)
assert not missing, f"Dataset missing required columns: {missing}"

df = raw.to_pandas()[["anchor", "positive"]].copy()
df["anchor"] = df["anchor"].map(normalize_text)
df["positive"] = df["positive"].map(normalize_text)

before = len(df)
df = df[(df["anchor"].str.len() > 0) & (df["positive"].str.len() > 0)].copy()
after_nonempty = len(df)

df["anchor_norm"] = df["anchor"].str.lower()
df["positive_norm"] = df["positive"].str.lower()

df = df.drop_duplicates(subset=["anchor_norm", "positive_norm"]).copy()
after_pair_dedup = len(df)

if DROP_DUPLICATE_ANCHORS:
    df = df.drop_duplicates(subset=["anchor_norm"], keep="first").copy()
after_anchor_dedup = len(df)

if DROP_DUPLICATE_POSITIVES:
    df = df.drop_duplicates(subset=["positive_norm"], keep="first").copy()
after_positive_dedup = len(df)

df["pair_hash"] = (df["anchor_norm"] + "\n" + df["positive_norm"]).map(stable_hash)

rng = np.random.default_rng(SEED)
indices = np.arange(len(df))
rng.shuffle(indices)

n_total = len(df)
n_train = int(n_total * TRAIN_RATIO)
n_dev = int(n_total * DEV_RATIO)
n_test = n_total - n_train - n_dev

train_idx = indices[:n_train]
dev_idx = indices[n_train:n_train+n_dev]
test_idx = indices[n_train+n_dev:]

train_df = df.iloc[train_idx][["anchor", "positive", "pair_hash"]].reset_index(drop=True)
dev_df = df.iloc[dev_idx][["anchor", "positive", "pair_hash"]].reset_index(drop=True)
test_df = df.iloc[test_idx][["anchor", "positive", "pair_hash"]].reset_index(drop=True)

# Leakage gates
train_hashes = set(train_df["pair_hash"])
dev_hashes = set(dev_df["pair_hash"])
test_hashes = set(test_df["pair_hash"])

assert train_hashes.isdisjoint(dev_hashes)
assert train_hashes.isdisjoint(test_hashes)
assert dev_hashes.isdisjoint(test_hashes)

train_ds = Dataset.from_pandas(train_df[["anchor", "positive"]], preserve_index=False)
dev_ds = Dataset.from_pandas(dev_df[["anchor", "positive"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["anchor", "positive"]], preserve_index=False)

dataset_audit = {
    "raw_rows": before,
    "after_nonempty": after_nonempty,
    "after_pair_dedup": after_pair_dedup,
    "after_anchor_dedup": after_anchor_dedup,
    "after_positive_dedup": after_positive_dedup,
    "train_rows": len(train_ds),
    "dev_rows": len(dev_ds),
    "test_rows": len(test_ds),
    "drop_duplicate_anchors": DROP_DUPLICATE_ANCHORS,
    "drop_duplicate_positives": DROP_DUPLICATE_POSITIVES,
    "seed": SEED,
}

print(json.dumps(dataset_audit, indent=2, ensure_ascii=False))
save_json(dataset_audit, METRICS_DIR / "dataset_audit.json")


In [ ]:

# 06_build_ir_evaluators

def build_ir_evaluator(eval_ds, all_corpus_ds, name):
    # Corpus gồm toàn bộ unique positives trong train+dev+test.
    # Đây là evaluator nội bộ để chọn checkpoint, không thay thế external benchmark.
    corpus_texts = list(dict.fromkeys(all_corpus_ds["positive"]))
    corpus = {f"doc-{i}": text for i, text in enumerate(corpus_texts)}
    text_to_cid = {text: cid for cid, text in corpus.items()}

    queries = {}
    relevant_docs = {}
    skipped = 0

    for i, ex in enumerate(eval_ds):
        qid = f"q-{i}"
        cid = text_to_cid.get(ex["positive"])
        if cid is None:
            skipped += 1
            continue
        queries[qid] = ex["anchor"]
        relevant_docs[qid] = {cid}

    assert len(queries) > 0, f"No queries built for evaluator {name}"
    assert skipped == 0, f"Skipped {skipped} examples because positive was not in corpus"

    evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        accuracy_at_k=[1, 3, 5, 10],
        precision_recall_at_k=[1, 3, 5, 10],
        mrr_at_k=[10],
        ndcg_at_k=[10],
        map_at_k=[100],
        score_functions={"cosine": cos_sim},
        main_score_function="cosine",
        show_progress_bar=True,
        batch_size=EVAL_BATCH_SIZE,
    )
    return evaluator, {"num_queries": len(queries), "num_corpus": len(corpus)}

all_corpus_ds = concatenate_datasets([train_ds, dev_ds, test_ds])

dev_evaluator, dev_eval_info = build_ir_evaluator(dev_ds, all_corpus_ds, "medical_dev")
test_evaluator, test_eval_info = build_ir_evaluator(test_ds, all_corpus_ds, "medical_test")

print("dev_eval_info:", dev_eval_info)
print("test_eval_info:", test_eval_info)

save_json(dev_eval_info, METRICS_DIR / "dev_eval_info.json")
save_json(test_eval_info, METRICS_DIR / "test_eval_info.json")


In [ ]:

# 07_baseline_eval_base_bge_m3

base_dev_metrics = {}
base_test_metrics = {}
METRIC_FOR_BEST_MODEL = None

if RUN_BASELINE_EVAL:
    cleanup_cuda()
    base_model = load_sentence_model(BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, trust_remote_code=True)
    print("Base model max_seq_length:", base_model.max_seq_length)

    _ = smoke_test_model(base_model, expected_dim=1024, name="base_model")

    base_dev_metrics = dev_evaluator(base_model)
    base_test_metrics = test_evaluator(base_model)

    print("BASE DEV METRICS")
    print(json.dumps(base_dev_metrics, indent=2, ensure_ascii=False))
    print("BASE TEST METRICS")
    print(json.dumps(base_test_metrics, indent=2, ensure_ascii=False))

    save_json(base_dev_metrics, METRICS_DIR / "base_dev_metrics.json")
    save_json(base_test_metrics, METRICS_DIR / "base_test_metrics.json")

    NDCG10_KEY = find_ndcg10_key(base_dev_metrics)
    METRIC_FOR_BEST_MODEL = f"eval_{NDCG10_KEY}" if not NDCG10_KEY.startswith("eval_") else NDCG10_KEY

    print("NDCG10_KEY:", NDCG10_KEY)
    print("METRIC_FOR_BEST_MODEL:", METRIC_FOR_BEST_MODEL)

    del base_model
    cleanup_cuda()
else:
    NDCG10_KEY = "medical_dev_cosine_ndcg@10"
    METRIC_FOR_BEST_MODEL = "eval_medical_dev_cosine_ndcg@10"
    print("Baseline skipped. Using metric key:", METRIC_FOR_BEST_MODEL)


In [ ]:

# 08_load_train_model_attach_lora_correctly

cleanup_cuda()

model = load_sentence_model(BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, trust_remote_code=True)
print("Train model loaded.")
print("max_seq_length:", model.max_seq_length)
print("module[0]:", type(model[0]))
print("auto_model before LoRA:", type(model[0].auto_model))

# Inspect target modules.
linear_names = []
for name, module in model[0].auto_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        linear_names.append(name)

matched = [
    name for name in linear_names
    if name.split(".")[-1] in set(LORA_TARGET_MODULES)
]

print("Number of Linear modules:", len(linear_names))
print("Number of matched LoRA target modules:", len(matched))
print("Sample matched modules:")
for n in matched[:40]:
    print("  ", n)

missing_target_kinds = []
for target in LORA_TARGET_MODULES:
    if not any(name.split(".")[-1] == target for name in linear_names):
        missing_target_kinds.append(target)

assert not missing_target_kinds, f"Missing LoRA target module kinds: {missing_target_kinds}"
assert len(matched) > 0, "No LoRA target modules matched. Check BGE-M3 module names."

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
)

# Critical fix:
# Ép wrap bằng PeftModelForFeatureExtraction để chắc chắn có merge_and_unload().
from peft import PeftModelForFeatureExtraction

base_auto_model = model[0].auto_model

peft_auto_model = PeftModelForFeatureExtraction(
    base_auto_model,
    peft_config,
    adapter_name="default",
)

model[0].auto_model = peft_auto_model

auto_model = model[0].auto_model
print("auto_model after PeftModelForFeatureExtraction:", type(auto_model))

assert hasattr(auto_model, "merge_and_unload"), (
    "PEFT wrapping failed: model[0].auto_model has no merge_and_unload(). "
    "Do not train/save/push because final artifact may become adapter-only."
)

if hasattr(auto_model, "print_trainable_parameters"):
    auto_model.print_trainable_parameters()

# PEFT + gradient checkpointing fix:
# Nếu chỉ LoRA trainable còn base frozen, cần ép input embeddings tham gia graph gradient.
if hasattr(auto_model, "enable_input_require_grads"):
    auto_model.enable_input_require_grads()
    print("Enabled input require grads for PEFT + gradient checkpointing.")
else:
    def make_inputs_require_grad(module, input, output):
        output.requires_grad_(True)

    input_embeddings = auto_model.get_input_embeddings()
    input_embeddings.register_forward_hook(make_inputs_require_grad)
    print("Registered input embedding forward hook for PEFT + gradient checkpointing.")

if hasattr(auto_model, "gradient_checkpointing_enable"):
    try:
        auto_model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        print("Gradient checkpointing enabled with use_reentrant=False.")
    except TypeError:
        auto_model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled.")

if hasattr(auto_model, "config") and hasattr(auto_model.config, "use_cache"):
    auto_model.config.use_cache = False
    print("Set use_cache=False.")

_ = smoke_test_model(model, expected_dim=1024, name="train_model_with_lora")


In [ ]:
# 09_training_arguments_and_trainer
# Sửa chính: không dùng load_best_model_at_end của Trainer nữa.
# Thay vào đó, dùng callback tự lưu LoRA adapter state tốt nhất theo dev NDCG@10.

train_loss = losses.MultipleNegativesRankingLoss(
    model=model,
    scale=20.0,
)

num_train_steps_per_epoch = math.ceil(len(train_ds) / TRAIN_BATCH_SIZE)
approx_total_steps = num_train_steps_per_epoch * NUM_EPOCHS

print("Approx steps/epoch:", num_train_steps_per_epoch)
print("Approx total steps:", approx_total_steps)

EVAL_STEPS = 10
LOGGING_STEPS = 2
BEST_METRIC_MIN_DELTA = 0.0


class BestPeftAdapterCallback(TrainerCallback):
    """Save best LoRA adapter state manually, avoiding Trainer checkpoint reload mismatch."""

    def __init__(
        self,
        st_model,
        metric_name,
        output_dir,
        greater_is_better=True,
        min_delta=0.0,
    ):
        self.st_model = st_model
        self.metric_name = metric_name
        self.output_dir = Path(output_dir)
        self.greater_is_better = greater_is_better
        self.min_delta = float(min_delta)
        self.best_score = None
        self.best_step = None
        self.best_epoch = None

    def _is_better(self, score):
        if score is None:
            return False
        if self.best_score is None:
            return True
        if self.greater_is_better:
            return score > self.best_score + self.min_delta
        return score < self.best_score - self.min_delta

    def on_train_begin(self, args, state, control, **kwargs):
        if self.output_dir.exists():
            shutil.rmtree(self.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        print("Manual best-adapter saving enabled.")
        print("Best metric:", self.metric_name)
        print("Best adapter dir:", self.output_dir)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        score = extract_metric_value(metrics, self.metric_name)
        print(f"[BestPeftAdapterCallback] step={state.global_step} epoch={state.epoch} score={score}")

        if self._is_better(score):
            self.best_score = float(score)
            self.best_step = int(state.global_step)
            self.best_epoch = float(state.epoch) if state.epoch is not None else None

            save_current_peft_adapter_state(
                self.st_model,
                self.output_dir,
                metric_name=self.metric_name,
                score=self.best_score,
                step=self.best_step,
                epoch=self.best_epoch,
                source="trainer_eval",
            )


training_args_kwargs = dict(
    output_dir=str(TRAIN_OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    # Do not use Trainer best-checkpoint reload. It mismatches PEFT-wrapped ST state_dict keys.
    load_best_model_at_end=False,

    # We do not need full Trainer checkpoints; the callback saves only best LoRA adapter state.
    save_strategy="no",

    logging_steps=LOGGING_STEPS,
    bf16=USE_BF16,
    fp16=USE_FP16,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    report_to=[],
    seed=SEED,
    dataloader_drop_last=False,
)

# ST/Transformers version compatibility: eval_strategy vs evaluation_strategy.
sig = inspect.signature(SentenceTransformerTrainingArguments)
if "eval_strategy" in sig.parameters:
    training_args_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in sig.parameters:
    training_args_kwargs["evaluation_strategy"] = "steps"
else:
    raise RuntimeError("Cannot find eval_strategy/evaluation_strategy in SentenceTransformerTrainingArguments signature.")

training_args_kwargs["eval_steps"] = EVAL_STEPS

# Some versions support auto_find_batch_size; keep disabled by default for reproducibility.
if "auto_find_batch_size" in sig.parameters:
    training_args_kwargs["auto_find_batch_size"] = False

args = SentenceTransformerTrainingArguments(**training_args_kwargs)
print(args)

best_adapter_callback = BestPeftAdapterCallback(
    st_model=model,
    metric_name=METRIC_FOR_BEST_MODEL,
    output_dir=BEST_ADAPTER_DIR,
    greater_is_better=True,
    min_delta=BEST_METRIC_MIN_DELTA,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    loss=train_loss,
    evaluator=dev_evaluator,
    callbacks=[best_adapter_callback],
)


In [ ]:
# 10_train_and_load_best_adapter
# Sau cell này, model trong RAM phải là best LoRA adapter theo dev cosine_ndcg@10.
# Không dùng load_best_model_at_end của Trainer.

train_result = trainer.train()
print(train_result)

trainer_state_path = METRICS_DIR / "trainer_state.json"
try:
    trainer.state.save_to_json(str(trainer_state_path))
    print("Saved trainer state:", trainer_state_path)
except Exception as e:
    print("Could not save trainer state:", repr(e))

# Evaluate final model once because eval_steps may not hit the exact final step.
# If the final model is better than the best saved during eval steps, save it as best.
final_train_dev_metrics = dev_evaluator(model)
final_train_dev_score = extract_metric_value(final_train_dev_metrics, METRIC_FOR_BEST_MODEL)

print("FINAL TRAIN DEV METRICS BEFORE BEST-ADAPTER LOAD")
print(json.dumps(final_train_dev_metrics, indent=2, ensure_ascii=False))
print("Final train dev score:", final_train_dev_score)

save_json(final_train_dev_metrics, METRICS_DIR / "final_train_dev_metrics_before_best_adapter_load.json")

best_info = None
if BEST_ADAPTER_INFO_PATH.exists():
    with BEST_ADAPTER_INFO_PATH.open("r", encoding="utf-8") as f:
        best_info = json.load(f)

best_score = None if best_info is None else best_info.get("score")
print("Best score saved during trainer eval:", best_score)

if final_train_dev_score is not None and (best_score is None or final_train_dev_score > float(best_score) + 1e-12):
    best_info = save_current_peft_adapter_state(
        model,
        BEST_ADAPTER_DIR,
        metric_name=METRIC_FOR_BEST_MODEL,
        score=final_train_dev_score,
        step=int(trainer.state.global_step),
        epoch=float(trainer.state.epoch) if trainer.state.epoch is not None else None,
        source="final_after_train",
    )

assert BEST_ADAPTER_STATE_PATH.exists(), (
    "No best adapter state was saved. Check evaluator metrics and callback."
)

with BEST_ADAPTER_INFO_PATH.open("r", encoding="utf-8") as f:
    best_info = json.load(f)

print("BEST ADAPTER INFO")
print(json.dumps(best_info, indent=2, ensure_ascii=False))

# Load best adapter state into current PEFT-wrapped model before merge.
model = load_peft_adapter_state_into_model(model, BEST_ADAPTER_STATE_PATH, adapter_name="default")

best_adapter_dev_metrics = dev_evaluator(model)
best_adapter_test_metrics = test_evaluator(model)

print("BEST ADAPTER DEV METRICS BEFORE MERGE")
print(json.dumps(best_adapter_dev_metrics, indent=2, ensure_ascii=False))
print("BEST ADAPTER TEST METRICS BEFORE MERGE")
print(json.dumps(best_adapter_test_metrics, indent=2, ensure_ascii=False))

save_json(best_adapter_dev_metrics, METRICS_DIR / "best_adapter_dev_metrics_before_merge.json")
save_json(best_adapter_test_metrics, METRICS_DIR / "best_adapter_test_metrics_before_merge.json")
save_json(best_info, METRICS_DIR / "best_adapter_info.json")

# Keep old variable names so downstream cells continue to work if referenced.
post_train_dev_metrics = best_adapter_dev_metrics
post_train_test_metrics = best_adapter_test_metrics

save_json(post_train_dev_metrics, METRICS_DIR / "post_train_dev_metrics_before_merge.json")
save_json(post_train_test_metrics, METRICS_DIR / "post_train_test_metrics_before_merge.json")

print("Best adapter loaded. Next cell will merge this best adapter into the base model.")


In [ ]:

# 11_merge_best_lora_adapter_into_base_model
# Cell trước đã load best adapter. Cell này merge best LoRA adapter vào base model.

model.eval()
cleanup_cuda()

auto_model = model[0].auto_model

print("auto_model type before merge:", type(auto_model))
assert hasattr(auto_model, "merge_and_unload"), (
    "Current auto_model has no merge_and_unload(). "
    "This means LoRA was not attached as a PEFT model. "
    "Do not save/push because output would not be a full merged model."
)

# Merge LoRA weights into base weights and remove LoRA modules.
merged_auto_model = auto_model.merge_and_unload()

# Important: assign back; do not assume merge is in-place.
model[0].auto_model = merged_auto_model

print("auto_model type after merge:", type(model[0].auto_model))

lora_modules = [
    name for name, _ in model[0].auto_model.named_modules()
    if "lora" in name.lower()
]

print("Remaining LoRA modules:", len(lora_modules))
if lora_modules[:20]:
    print(lora_modules[:20])

assert len(lora_modules) == 0, "LoRA modules still exist after merge. Do not save/push."

_ = smoke_test_model(model, expected_dim=1024, name="merged_model_in_ram")


In [ ]:

# 12_save_full_merged_sentence_transformers_artifact

if LOCAL_MERGED_DIR.exists():
    shutil.rmtree(LOCAL_MERGED_DIR)

LOCAL_MERGED_DIR.mkdir(parents=True, exist_ok=True)

# Save bằng SentenceTransformers format, không dùng Unsloth save_pretrained_merged.
try:
    model.save(str(LOCAL_MERGED_DIR), safe_serialization=True)
except TypeError:
    model.save(str(LOCAL_MERGED_DIR))

print("Saved full merged SentenceTransformers artifact to:", LOCAL_MERGED_DIR)

all_files = [p.relative_to(LOCAL_MERGED_DIR).as_posix() for p in LOCAL_MERGED_DIR.rglob("*") if p.is_file()]
print("Saved files:")
for f in all_files[:250]:
    print(" ", f)
if len(all_files) > 250:
    print(" ...", len(all_files) - 250, "more files")

assert (LOCAL_MERGED_DIR / "modules.json").exists(), "Missing modules.json; not a valid SentenceTransformers artifact."
assert (LOCAL_MERGED_DIR / "config_sentence_transformers.json").exists(), "Missing config_sentence_transformers.json."

adapter_files = [f for f in all_files if "adapter_model" in f or "adapter_config" in f]
assert len(adapter_files) == 0, "Adapter files found in merged output: " + str(adapter_files)

weight_files = [
    f for f in all_files
    if re.search(r"(model|pytorch_model).*\.(safetensors|bin)$", f)
]
print("Weight files:", weight_files)
assert len(weight_files) > 0, "No full model weight files found. Do not push."

modules_json_text = (LOCAL_MERGED_DIR / "modules.json").read_text(encoding="utf-8")
assert "unsloth" not in modules_json_text.lower(), "modules.json contains Unsloth reference."
assert "sentence_transformers.base" not in modules_json_text, "modules.json contains old broken sentence_transformers.base reference."

print("Full merged artifact file gate passed.")


In [ ]:

# 13_local_reload_gate_and_eval
# Nếu cell này fail thì không được push Hub.

cleanup_cuda()

local_reloaded = load_sentence_model(str(LOCAL_MERGED_DIR), max_seq_length=MAX_SEQ_LENGTH, trust_remote_code=False)

_ = smoke_test_model(local_reloaded, expected_dim=1024, name="local_reloaded_model")

local_dev_metrics = dev_evaluator(local_reloaded)
local_test_metrics = test_evaluator(local_reloaded)

print("LOCAL RELOADED DEV METRICS")
print(json.dumps(local_dev_metrics, indent=2, ensure_ascii=False))
print("LOCAL RELOADED TEST METRICS")
print(json.dumps(local_test_metrics, indent=2, ensure_ascii=False))

save_json(local_dev_metrics, METRICS_DIR / "local_reloaded_dev_metrics_after_merge.json")
save_json(local_test_metrics, METRICS_DIR / "local_reloaded_test_metrics_after_merge.json")

print("Local reload gate passed.")

# Replace current model reference with local reloaded one to confirm subsequent metadata uses reloadable artifact.
del model
model = local_reloaded
cleanup_cuda()


In [ ]:

# 14_write_model_card_and_training_audit

base_dev_summary = summarize_ir_metrics(base_dev_metrics) if base_dev_metrics else {}
base_test_summary = summarize_ir_metrics(base_test_metrics) if base_test_metrics else {}
local_dev_summary = summarize_ir_metrics(local_dev_metrics)
local_test_summary = summarize_ir_metrics(local_test_metrics)

best_adapter_info = {}
if BEST_ADAPTER_INFO_PATH.exists():
    with BEST_ADAPTER_INFO_PATH.open('r', encoding='utf-8') as f:
        best_adapter_info = json.load(f)

versions = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "peft": peft.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
}

audit_summary = {
    "base_model": BASE_MODEL,
    "dataset_id": DATASET_ID,
    "repo_id": REPO_ID,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "dataset_audit": dataset_audit,
    "dev_eval_info": dev_eval_info,
    "test_eval_info": test_eval_info,
    "metric_for_best_model": METRIC_FOR_BEST_MODEL,
    "best_adapter_info": best_adapter_info,
    "local_merged_dir": str(LOCAL_MERGED_DIR),
    "pushed_to_hub": PUSH_TO_HUB,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "versions": versions,
}

save_json(audit_summary, METRICS_DIR / "final_audit_summary.json")

model_card = f'''---
language:
- vi
license: apache-2.0
base_model: {BASE_MODEL}
library_name: sentence-transformers
pipeline_tag: sentence-similarity
tags:
- sentence-transformers
- feature-extraction
- embeddings
- vietnamese
- medical
- bge-m3
- lora
---

# BGE-M3 Medical Vietnamese Dense Embedding

This model is a dense SentenceTransformers embedding model fine-tuned from `{BASE_MODEL}` on `{DATASET_ID}`.

## Important

This repository is intended to be loadable directly with:

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("{REPO_ID}")
```

The model was trained with PEFT LoRA. The best LoRA adapter was selected manually by dev cosine NDCG@10, then merged into the base weights before saving. The final uploaded artifact should not be adapter-only.

## Best adapter selection

The best adapter is selected manually because `Trainer.load_best_model_at_end` can fail with PEFT-wrapped SentenceTransformers checkpoints due to state_dict key prefix mismatch.

```json
{json.dumps(best_adapter_info, ensure_ascii=False, indent=2)}
```

## Training setup

- Base model: `{BASE_MODEL}`
- Dataset: `{DATASET_ID}`
- Loss: `MultipleNegativesRankingLoss(scale=20.0)`
- Max sequence length: `{MAX_SEQ_LENGTH}`
- Epochs: `{NUM_EPOCHS}`
- Batch size: `{TRAIN_BATCH_SIZE}`
- Gradient accumulation: `{GRADIENT_ACCUMULATION_STEPS}`
- Learning rate: `{LEARNING_RATE}`
- Warmup ratio: `{WARMUP_RATIO}`
- Weight decay: `{WEIGHT_DECAY}`
- LoRA r: `{LORA_R}`
- LoRA alpha: `{LORA_ALPHA}`
- LoRA dropout: `{LORA_DROPOUT}`
- LoRA target modules: `{LORA_TARGET_MODULES}`

## Scope

This is a dense retrieval fine-tune. It should not be described as explicitly fine-tuning the sparse or multi-vector components of BGE-M3.

## Internal evaluation

Evaluator: `InformationRetrievalEvaluator`
Main metric: cosine NDCG@10
Split: train/dev/test = 90/5/5 after cleaning and exact deduplication.

### Local merged metrics

Dev:

```json
{json.dumps(local_dev_summary, ensure_ascii=False, indent=2)}
```

Test:

```json
{json.dumps(local_test_summary, ensure_ascii=False, indent=2)}
```

### Base model metrics

Dev:

```json
{json.dumps(base_dev_summary, ensure_ascii=False, indent=2)}
```

Test:

```json
{json.dumps(base_test_summary, ensure_ascii=False, indent=2)}
```

## Limitations

- This model is not a medical diagnosis system.
- Retrieval results should be reviewed before any clinical or safety-critical use.
- Internal evaluation is not a substitute for external benchmarks.
- Dataset distribution may not represent all Vietnamese medical domains or real-world user queries.
'''

readme_path = LOCAL_MERGED_DIR / "README.md"
readme_path.write_text(model_card, encoding="utf-8")
print("Wrote model card:", readme_path)

# Copy metrics into artifact folder.
artifact_metrics_dir = LOCAL_MERGED_DIR / "training_metrics"
artifact_metrics_dir.mkdir(exist_ok=True)
for p in METRICS_DIR.glob("*.json"):
    shutil.copy2(p, artifact_metrics_dir / p.name)

print("Copied metrics into:", artifact_metrics_dir)


In [ ]:

# 15_push_full_merged_model_to_hub
# Chỉ push folder full merged đã pass local reload gate.

if PUSH_TO_HUB:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

    upload_kwargs = dict(
        folder_path=str(LOCAL_MERGED_DIR),
        repo_id=REPO_ID,
        repo_type="model",
        commit_message="Upload full merged SentenceTransformers BGE-M3 medical VI dense model",
    )

    if DELETE_REMOTE_FILES_BEFORE_UPLOAD:
        upload_folder_sig = inspect.signature(api.upload_folder)
        if "delete_patterns" not in upload_folder_sig.parameters:
            raise RuntimeError(
                "Installed huggingface_hub.HfApi.upload_folder does not support delete_patterns. "
                "Upgrade huggingface_hub or manually clear the target repo before uploading."
            )
        upload_kwargs["delete_patterns"] = ["*"]

    api.upload_folder(**upload_kwargs)
    print("Uploaded full merged model to Hub:", REPO_ID)
else:
    print("PUSH_TO_HUB=False. Skipped upload.")


In [ ]:

# 16_hub_reload_gate_and_final_eval
# Gate cuối: nếu cell này pass thì repo public load được bằng SentenceTransformer.

if PUSH_TO_HUB:
    cleanup_cuda()

    if HUB_RELOAD_CACHE.exists():
        shutil.rmtree(HUB_RELOAD_CACHE)
    HUB_RELOAD_CACHE.mkdir(parents=True, exist_ok=True)

    try:
        hub_model = SentenceTransformer(
            REPO_ID,
            cache_folder=str(HUB_RELOAD_CACHE),
            model_kwargs={"dtype": TORCH_DTYPE},
            trust_remote_code=False,
        )
    except TypeError:
        hub_model = SentenceTransformer(
            REPO_ID,
            cache_folder=str(HUB_RELOAD_CACHE),
            trust_remote_code=False,
        )
        if TORCH_DTYPE != torch.float32:
            hub_model = hub_model.to(TORCH_DTYPE)

    hub_model.max_seq_length = MAX_SEQ_LENGTH

    _ = smoke_test_model(hub_model, expected_dim=1024, name="hub_reloaded_model")

    hub_dev_metrics = dev_evaluator(hub_model)
    hub_test_metrics = test_evaluator(hub_model)

    print("HUB RELOADED DEV METRICS")
    print(json.dumps(hub_dev_metrics, indent=2, ensure_ascii=False))
    print("HUB RELOADED TEST METRICS")
    print(json.dumps(hub_test_metrics, indent=2, ensure_ascii=False))

    save_json(hub_dev_metrics, METRICS_DIR / "hub_reloaded_dev_metrics.json")
    save_json(hub_test_metrics, METRICS_DIR / "hub_reloaded_test_metrics.json")

    print("Hub reload gate passed.")
else:
    print("PUSH_TO_HUB=False. Skipped Hub reload gate.")


In [ ]:

# 17_export_zip_for_download

if EXPORT_ZIP_PATH.exists():
    EXPORT_ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(EXPORT_ZIP_PATH).replace(".zip", ""),
    format="zip",
    root_dir=str(LOCAL_MERGED_DIR),
)

print("Exported zip:", EXPORT_ZIP_PATH)
print("Notebook completed.")
